## Objective

This notebook focuses on data modeling for Olist E-Commerce Sales Analysis.  
It merges and aggregates relevant datasets to create a **fact table** (`fact_sales`) and **dimension tables** (`dim_customers`, `dim_sellers`).  
Feature engineering is performed to calculate metrics like **delivery time**, **revenue**, and extract **order month/year**, preparing clean structured tables ready for analysis and visualization in Power BI.

In [29]:
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
customers = pd.read_csv("/content/drive/MyDrive/original/customers_clean.csv")
orders = pd.read_csv("/content/drive/MyDrive/original/orders_clean.csv")
items = pd.read_csv("/content/drive/MyDrive/original/items_clean.csv")
payments = pd.read_csv("/content/drive/MyDrive/original/payments_clean.csv")
reviews = pd.read_csv("/content/drive/MyDrive/original/reviews_clean.csv")
sellers = pd.read_csv("/content/drive/MyDrive/original/sellers_clean.csv")
products = pd.read_csv("/content/drive/MyDrive/original/products_clean.csv")
name_translations = pd.read_csv("/content/drive/MyDrive/original/name_translations_clean.csv")

In [32]:
print("Missing values in customers DataFrame:")
print(customers.isnull().sum())
print("\nMissing values in orders DataFrame:")
print(orders.isnull().sum())
print("\nMissing values in items DataFrame:")
print(items.isnull().sum())
print("\nMissing values in payments DataFrame:")
print(payments.isnull().sum())
print("\nMissing values in reviews DataFrame:")
print(reviews.isnull().sum())
print("\nMissing values in sellers DataFrame:")
print(sellers.isnull().sum())
print("\nMissing values in products DataFrame:")
print(products.isnull().sum())
print("\nMissing values in name_translations DataFrame:")
print(name_translations.isnull().sum())

Missing values in customers DataFrame:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Missing values in orders DataFrame:
order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64

Missing values in items DataFrame:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Missing values in payments DataFrame:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

Missing values in reviews DataFrame:
review_id               

In [34]:
data = customers.merge(orders, on = 'customer_id').merge(items, on = 'order_id').merge(payments, on = 'order_id') \
                .merge(reviews, on = 'order_id').merge(products, on = 'product_id').merge(name_translations, on = 'product_category_name') \
                .merge(sellers, on = 'seller_id')
data.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,...,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,...,1141.0,1.0,8683.0,54.0,64.0,31.0,office_furniture,8577,itaquaquecetuba,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,...,1002.0,3.0,10150.0,89.0,15.0,40.0,housewares,88303,itajai,SC
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,...,955.0,1.0,8267.0,52.0,52.0,17.0,office_furniture,8577,itaquaquecetuba,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,...,1066.0,1.0,12160.0,56.0,51.0,28.0,office_furniture,8577,itaquaquecetuba,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,...,407.0,1.0,5200.0,45.0,15.0,35.0,home_confort,14940,ibitinga,SP


In [42]:
dim_customers = customers[['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']]

In [43]:
print("dim_customers Head:")
print(dim_customers.head())

dim_customers Head:
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  


In [44]:
print("dim_customers Info:")
dim_customers.info()

dim_customers Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [45]:
dim_sellers = sellers[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']]

In [46]:
print("dim_sellers Head:")
print(dim_sellers.head())

dim_sellers Head:
                          seller_id  seller_zip_code_prefix  \
0  3442f8959a84dea7ee197c632cb2df15                   13023   
1  d1b65fc7debc3361ea86b5f14c68d2e2                   13844   
2  ce3ad9de960102d0677a81f5d0bb7b2d                   20031   
3  c0f3eea2e14555b6faeea3dd58c1b1c3                    4195   
4  51a04a8a6bdcb23deccc82b0b80742cf                   12914   

         seller_city seller_state  
0           campinas           SP  
1         mogi guacu           SP  
2     rio de janeiro           RJ  
3          sao paulo           SP  
4  braganca paulista           SP  


In [47]:
print("dim_sellers Info:")
dim_sellers.info()

dim_sellers Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


In [48]:
data['order_purchase_timestamp'] = pd.to_datetime(data['order_purchase_timestamp'])
data['order_delivered_customer_date'] = pd.to_datetime(data['order_delivered_customer_date'])

In [49]:
data['delivery_time_days'] = (data['order_delivered_customer_date'] - data['order_purchase_timestamp']).dt.days
data['revenue'] = data['price'] + data['freight_value']
data['order_month'] = data['order_purchase_timestamp'].dt.month
data['order_year'] = data['order_purchase_timestamp'].dt.year

In [50]:
fact_sales = data[['order_id', 'customer_id', 'seller_id', 'product_id', 'payment_type', 'review_score', 'price', 'freight_value', 'revenue', 'delivery_time_days','order_purchase_timestamp', 'order_month', 'order_year']]

In [51]:
print("fact_sales Head:")
fact_sales.head()

fact_sales Head:


,order_id,customer_id,seller_id,product_id,payment_type,review_score,price,freight_value,revenue,delivery_time_days,order_purchase_timestamp,order_month,order_year
0,00e7ee1b050b8499577073aeb2a297a1,06b8999e2fba1a1fbc88172c00ba8bc7,7c67e1448b00f6e969d365cea6b010ab,a9516a079e37a9c9c36b9b78b10169e8,credit_card,4,124.99,21.88,146.87,8,2017-05-16 15:05:35,5,2017
1,29150127e6685892b6eab3eec79f59c7,18955e83d337fd6b2def6b18a428ac77,b8bc237ba3788b23da09c0f1f3a3288c,4aa6014eceb682077f9dc4bffebc05b0,credit_card,5,289.00,46.48,335.48,16,2018-01-12 20:48:24,1,2018
2,b2059ed67ce144a36e2aa97d2c9e9ad2,4e7b3e00288586ebd08712fdd0374a03,7c67e1448b00f6e969d365cea6b010ab,bd07b66896d6f1494f5b86251848ced7,credit_card,5,139.94,17.79,157.73,26,2018-05-19 16:07:45,5,2018
3,951670f92359f4fe4a63112aa7306eba,b2b6027bc5c5109e529d4dc6358b12c3,7c67e1448b00f6e969d365cea6b010ab,a5647c44af977b148e0a3a4751a09e2e,credit_card,5,149.94,23.36,173.30,14,2018-03-13 16:06:38,3,2018
4,6b7d50bd145f6fc7f33cebabd7e49d0f,4f2d8ab171c80ec8364f7c12e35b23ad,4a3ca9315b744ce9f8e9374361493884,9391a573abe00141c56e38d84d7d5b3b,credit_card,5,230.00,22.25,252.25,11,2018-07-29 09:51:30,7,2018


In [52]:
print("fact_sales Info:")
fact_sales.info()

fact_sales Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113186 entries, 0 to 113185
Data columns (total 13 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   order_id                  113186 non-null  object        
 1   customer_id               113186 non-null  object        
 2   seller_id                 113186 non-null  object        
 3   product_id                113186 non-null  object        
 4   payment_type              113186 non-null  object        
 5   review_score              113186 non-null  int64         
 6   price                     113186 non-null  float64       
 7   freight_value             113186 non-null  float64       
 8   revenue                   113186 non-null  float64       
 9   delivery_time_days        113186 non-null  int64         
 10  order_purchase_timestamp  113186 non-null  datetime64[ns]
 11  order_month               113186 non-null  int32

In [41]:
dim_customers.to_csv('/content/dim_customers.csv', index=False)
dim_sellers.to_csv('/content/dim_sellers.csv', index=False)
fact_sales.to_csv('/content/fact_sales.csv', index=False)

print("dim_customers.csv, dim_sellers.csv, and fact_sales.csv have been saved.")

dim_customers.csv, dim_sellers.csv, and fact_sales.csv have been saved.
